# CLAP Music Playlist Generator

Notebook pour generer une playlist a partir d'un prompt texte avec CLAP (LAION).
Ce notebook charge le checkpoint musical `music_audioset_epoch_15_esc_90.14.pt`.

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

# -----------------------------
# Parametres faciles a modifier
# -----------------------------
MUSIC_REPO_PATH = Path("Songs")  # dossier de musiques (ou dossier contenant des sous-dossiers)
PROMPT = "energetic club track with a strong bassline"
PLAYLIST_SIZE = 5

# Checkpoint CLAP music (cf README LAION-AI CLAP)
CHECKPOINT_URL = "https://huggingface.co/lukewys/laion_clap/resolve/main/music_audioset_epoch_15_esc_90.14.pt"
CHECKPOINT_PATH = Path("checkpoints/music_audioset_epoch_15_esc_90.14.pt")

# Optionnel
BATCH_SIZE = 8

In [3]:
import urllib.request
import numpy as np
import torch
import laion_clap

AUDIO_EXTS = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".aac", ".aiff", ".wma"}
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

def ensure_checkpoint(checkpoint_path: Path, checkpoint_url: str) -> Path:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    if not checkpoint_path.exists():
        print(f"Downloading checkpoint to {checkpoint_path}...")
        urllib.request.urlretrieve(checkpoint_url, checkpoint_path)
    else:
        print(f"Using existing checkpoint: {checkpoint_path}")
    return checkpoint_path

def collect_audio_files(root: Path):
    if not root.exists():
        raise FileNotFoundError(f"Music path does not exist: {root}")
    files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS]
    return sorted(files)

def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + eps)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# This step lasts around 1m10s

ckpt_file = ensure_checkpoint(CHECKPOINT_PATH, CHECKPOINT_URL)

model = laion_clap.CLAP_Module(enable_fusion=False, amodel="HTSAT-base", device=DEVICE)
model.load_ckpt(str(ckpt_file))
print(f"Model loaded on {DEVICE}: {ckpt_file.name}")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4319.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load the specified checkpoint checkpoints/music_audioset_epoch_15_esc_90.14.pt from users.
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.bl

In [8]:
audio_files = collect_audio_files(MUSIC_REPO_PATH)
if not audio_files:
    raise RuntimeError(f"No audio files found in: {MUSIC_REPO_PATH}")

print(f"Found {len(audio_files)} audio file(s) under: {MUSIC_REPO_PATH}")
for i, f in enumerate(audio_files[:10], start=1):
    print(f"{i:02d}. {f}")
if len(audio_files) > 10:
    print(f"... and {len(audio_files) - 10} more")

audio_paths = [str(p) for p in audio_files]

audio_embs_chunks = []
for start in range(0, len(audio_paths), BATCH_SIZE):
    batch_paths = audio_paths[start:start + BATCH_SIZE]
    batch_embs = model.get_audio_embedding_from_filelist(x=batch_paths, use_tensor=False)
    audio_embs_chunks.append(np.asarray(batch_embs, dtype=np.float32))

audio_embs = np.vstack(audio_embs_chunks)
audio_embs = l2_normalize(audio_embs)
print("Audio embeddings shape:", audio_embs.shape)

Found 33 audio file(s) under: Songs
01. Songs/BXTR - Neo Paradise Circus [RAWSH02].mp3
02. Songs/CHRIS _P - 1, 2, 3.mp3
03. Songs/Catching Feelings (Edit).mp3
04. Songs/DIAMS - DJ ( URUMI RAVE REMIX ) [Free dl].mp3
05. Songs/DKR (SubLife x Jaykill Remix).mp3
06. Songs/Dr. G - RENEGADE MASTER [regroove].mp3
07. Songs/Dr. G x Helena Lauwaert - Girl.mp3
08. Songs/Dreamstate (feat. Katy B).mp3
09. Songs/Funk Tribu - DR34M$ (extended mix).mp3
10. Songs/Hamza Ft. Damso - God Bless (Mangabey Remix).mp3
... and 23 more
Audio embeddings shape: (33, 512)


In [9]:
def generate_playlist(prompt: str, playlist_size: int = 5):
    if playlist_size <= 0:
        raise ValueError("playlist_size must be > 0")

    text_emb = model.get_text_embedding([prompt], use_tensor=False)
    text_emb = l2_normalize(np.asarray(text_emb, dtype=np.float32))[0]

    scores = audio_embs @ text_emb
    top_k = min(playlist_size, len(audio_files))
    ranked_idx = np.argsort(-scores)[:top_k]

    return [(audio_files[i], float(scores[i])) for i in ranked_idx]

playlist = generate_playlist(PROMPT, PLAYLIST_SIZE)

print("Prompt:", PROMPT)
print(f"Playlist size requested: {PLAYLIST_SIZE}")
print("\nGenerated playlist:")
for rank, (track_path, score) in enumerate(playlist, start=1):
    print(f"{rank:02d}. {score: .4f} | {track_path}")

Prompt: energetic club track with a strong bassline
Playlist size requested: 5

Generated playlist:
01.  0.4287 | Songs/CHRIS _P - 1, 2, 3.mp3
02.  0.4237 | Songs/Haus Muzik.mp3
03.  0.4173 | Songs/Dr. G x Helena Lauwaert - Girl.mp3
04.  0.4141 | Songs/DIAMS - DJ ( URUMI RAVE REMIX ) [Free dl].mp3
05.  0.3998 | Songs/The Process Is The Good Part.mp3
